# Interface of energy transition model with hydrogen 

The hydrogen demand is served by SMR (grey), SMR + CCS (blue) and the grid electrolyser (green). Hydrogen storage, the pure green route and pink hydrogen are in the package with additions, which runs the same analysis with them in every run.

- **Hydrogen is a demand.** The demand table has one new demand, `industrial_h2`: 1.3 Mt/yr for refineries and ammonia (Elzenga et al., 2025). It is the same in every period and scenario, and flat over the hours.

- **End-use technologies feed that demand directly:** SMR (grey hydrogen), SMR + CCS (blue) and the electrolyser (green, grid electricity).
  The reformers that exist in 2020 are the 2020 stock (`SMR - existing`).
  
- **The scenarios differ only in water availability.** Water itself is not modelled. The scenario limits how much electrolyser capacity can be built, in the capacity constraints of the technology table. **The base is the scenario without water scarcity.**

## 0. Imports and settings

In [ ]:
import os, json, sys, warnings
from pathlib import Path

# Make the notebook independent of the folder from which VS Code/Jupyter starts the kernel.
def _find_model_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for parent in candidates:
        options = [parent] + [p for p in parent.iterdir() if p.is_dir()] if parent.exists() else []
        for option in options:
            if (option / 'interface.ipynb').is_file() and (option / 'energy_model').is_dir():
                return option
    raise RuntimeError('Could not find the FinalModel_H2 folder. Open this notebook from the published model checkout.')

MODEL_ROOT = _find_model_root()
os.chdir(MODEL_ROOT)
if str(MODEL_ROOT) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT))
warnings.filterwarnings('ignore')
import logging
logging.getLogger('pypsa').setLevel(logging.ERROR)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pypsa
from IPython.display import Image, display

from energy_model import h2_presets as h2
from energy_model.h2_scenario import electrolyser_reference


from figures.plots.h2_results import (summary, compare, policy_table, h2_balance,
                                       limit_cost, cumulative_co2, subsidy_cost,
                                       gas_use, electricity_mix, co2_stored_by_sector)


from figures.plots import h2_plots as hp
from figures.plots.h2_sankey import plot_sankeys, plot_sankeys_row
from figures.plots import system_plots as sp
from figures.plots import presentation as pres
ADDITIONS, SET_LONG = False, 'Published FinalModel_H2'
import run_batch                      # list of runs (RUNS) and a cached run() function

pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
pd.set_option('display.max_columns', 40)

SOLVER = 'gurobi'
FIG = 'results/plots'
RERUN = False          # True = re-solve even if results/runs/<name>.nc already exists
os.makedirs(FIG, exist_ok=True)
SHORT = h2.SHORT_NAMES          # {'S1': 'Base', 'S2': 'Medium', 'S3': 'Scarce'}
print('Package', SET_LONG)

def load(name):
    n = pypsa.Network(run_batch.run(name, solver=SOLVER, rerun=RERUN))
    n.name = name
    return n

def save(fig, name):
    fig = getattr(fig, 'figure', fig)
    for ext in ('png', 'svg'):
        fig.savefig(f'{FIG}/{name}.{ext}', dpi=300, bbox_inches='tight', facecolor='white')

## 1. Importing the course parameters and plots

In [ ]:
# Default parameters 

db_dir = 'input_databases//h2_nl'     # hydrogen database (real numbers); 'input_databases//default' = course model
years = [2020, 2030, 2040, 2050]      # model years
t_res = 15                            # intra-year temporal resolution (hours)
rt = 0.07                             # technical discount rate (annuities)
rs = 0.01                             # social discount rate (weights of the years in the objective)
passenger_cars_cf = 40 * 0.1          # passenger cars conversion factor (km/h)

In [ ]:
# Handy plotting functions
from figures.plots.system_plots import course_plot_main_ebs as plot_main_ebs, course_plot_elc_emi as plot_elc_emi
from figures.plots.system_plots import course_plot_h2_eb as plot_h2_eb

# 2. Choosing the scenario 

In [ ]:
SCENARIO = 'S1'                       # 'S1' Base, 'S2' Medium, 'S3' Scarce

**The selected scenario**  loads the saved run of `SCENARIO` and draws the course plots. If the run is not on disk, it is solved, together with Base without additions if it needs its electrolyser capacity. The last plot is the balance of the hydrogen demand bus: the technologies on one side, the demand on the other.

In [ ]:
n = load(SCENARIO)
print(SCENARIO, SHORT[SCENARIO], '| objective:', f'{n.objective:.4e}')
if plot_main_ebs is not None:
    plot_main_ebs(n)
    plot_elc_emi(n)
    plot_h2_eb(n)
else:
    print('Core model loaded; optional course plotting modules are not included in this package.')

## 3. The hydrogen database

`build_h2_database.py` starts from the final course database (`input_databases/default`) and writes **one** hydrogen database, `h2_nl`. It contains:
- the final model's database, with its 2020 calibration unchanged;
- the hydrogen demand and the three technologies;
- the electrolyser on desalinated seawater (a green lever, off unless a run switches it on);
- the selected Dutch input values and their cited sources.

This package has no additions: the database has no hydrogen storage and no dedicated plants.

In [ ]:
import build_h2_database
REBUILD = False                 # True = rebuild input_databases/h2_nl (after editing build_h2_database.py)
if REBUILD or not os.path.exists('input_databases/h2_nl/technologies.xlsx'):
    build_h2_database.main()
s = pd.read_excel('input_databases/h2_nl/technologies.xlsx', sheet_name='static_parameters')
s['Technology'] = s['Technology'].ffill()
techs = ['SMR', 'SMR - existing', 'SMR CCS', 'Electrolyser', 'Electrolyser - seawater', 'H2 storage',
         'Solar PV - dedicated', 'Wind turbine - offshore - dedicated', 'Electrolyser - dedicated',
         'Nuclear plant - dedicated', 'Electrolyser - nuclear']
techs = [t for t in techs if t in set(s.Technology)]
s[s.Technology.isin(techs)].pivot_table(index='Technology', columns='Parameter', values='Value', aggfunc='first').reindex(
    columns=['Sector', 'Type', 'bus0', 'bus1', 'bus2', 'bus3', 'Start year']).loc[techs]

In [ ]:
d = pd.read_excel('input_databases/h2_nl/technologies.xlsx', sheet_name='dynamic_parameters')
d['Technology'] = d['Technology'].ffill()
d[d.Technology.isin(['SMR', 'SMR CCS', 'Electrolyser', 'H2 storage'])].set_index(['Technology', 'Parameter'])[
    ['Unit', 2020, 2030, 2040, 2050]]

In [ ]:
dem = pd.read_excel('input_databases/h2_nl/demand.xlsx', sheet_name='demands')
display(dem[dem.Demand == 'industrial_h2'][['Demand', 'Bus', 'Unit', 'Start_value', 'Exponent']])
par = pd.read_excel('input_databases/h2_nl/supply.xlsx', sheet_name='resources_parameters')
par[par.Parameter == 'Price'].set_index(['Resource', 'Unit'])[[2020, 2030, 2040, 2050]]

## 4. The three scenarios

**Base without additions sets the water.** Its electrolyser capacity per vintage sets the limit of Medium (50%); Scarce has no electrolysers. The limits go into the capacity constraints of the technology table, the course's own way of limiting how much of a technology can be built.

**In every run the limits stay the same in MW.** The water that is available does not change when costs change or technologies are added. With more than one electrolyser technology, one joint constraint keeps them together under the limit.

In [ ]:
runs = {SHORT[s]: load(s) for s in ['S1', 'S2', 'S3']}
water_ref = electrolyser_reference(load(run_batch.BASE_NOADD))
print('Electrolyser capacity of Base without additions per vintage (MW of hydrogen output):', json.dumps(water_ref))
print('Limit of Medium per vintage (MW of hydrogen output, all electrolysers together):',
      json.dumps(runs['Medium'].meta['scenario'].get('_electrolyser_cap_total_mw_h2')
                 or runs['Medium'].meta['scenario']['_electrolyser_caps_mw_h2']))

In [ ]:
if compare is not None:
    cols = ['Green (grid electrolysis)', 'Pure green (dedicated wind + solar)', 'Pink (nuclear electrolysis)',
            'Blue (SMR + CCS)', 'Grey (SMR)', 'Electrolysis share (%)', 'Electrolyser capacity (GW_el)',
            'Electrolyser full-load hours (h/yr)', 'H2 storage (GWh)', 'Gas use, total (TWh/yr)', 'Gas for H2 (TWh/yr)',
            'CO2 total (Mt/yr)', 'CO2 captured from H2 (Mt/yr)', 'Water for H2 (Mm3/yr)', 'System cost (bn €/yr)']
    pd.concat({p: compare(runs, period=p)[cols] for p in [2030, 2040, 2050]})
else:
    print('figures.plots.h2_results not available — skipping comparison table')

In [ ]:
import os
print(os.path.exists('figures/plots/h2_results.py'))
print(os.listdir('figures/plots') if os.path.exists('figures/plots') else 'figures/plots not found')

In [ ]:
from figures.plots.h2_results import compare


In [ ]:
print(os.path.exists('figures/__init__.py'))
print(os.path.exists('figures/plots/__init__.py'))

In [ ]:
open('figures/__init__.py', 'w').close()
open('figures/plots/__init__.py', 'w').close()

In [ ]:
cols = ['Green (grid electrolysis)', 'Pure green (dedicated wind + solar)', 'Pink (nuclear electrolysis)',
        'Blue (SMR + CCS)', 'Grey (SMR)', 'Electrolysis share (%)', 'Electrolyser capacity (GW_el)',
        'Electrolyser full-load hours (h/yr)', 'H2 storage (GWh)', 'Gas use, total (TWh/yr)', 'Gas for H2 (TWh/yr)',
        'CO2 total (Mt/yr)', 'CO2 captured from H2 (Mt/yr)', 'Water for H2 (Mm3/yr)', 'System cost (bn €/yr)']
pd.concat({p: compare(runs, period=p)[cols] for p in [2030, 2040, 2050]})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
hp.plot_h2_routes(runs, ax=ax, title='Hydrogen supply by route and period (TWh per year)')
save(ax, 'T1_1_h2_routes')

In [ ]:
ax = hp.plot_electrolyser_duration(runs, 2050)
save(ax, 'T1_1_electrolyser_duration_2050')

**Scarce at 0% or 10%?** The pessimistic case "can be zero, it can be ten percent". The report uses 0%, because electrolysis gets no water in a drought. `S3_10pct` allows 10% of the capacity of Base without additions.

In [ ]:
sc = {'Scarce (0%)': runs['Scarce'], 'Scarce (10%)': load('S3_10pct'), 'Medium (50%)': runs['Medium']}
policy_table(sc, ref='Scarce (0%)')

**How to read this.**
- **2030:** the existing reformers (grey) make all the hydrogen in every scenario.
- **2040:** blue hydrogen, because it is still cheaper than green.
- **2050:** green hydrogen is cheapest whenever electricity is cheap.
  - Without hydrogen storage and with a flat demand, the electrolysers can never usefully be larger than the demand itself. Base builds exactly that, and blue hydrogen fills the hours with expensive electricity.
  - Medium can build half of that capacity, and Scarce none.

## 5. The policy story

Compare the scenarios on what they mean for policymakers:
- the cost of the system;
- the gas that has to be imported;
- the electricity generation mix;
- the money saved by building a smaller electrolyser.

In [ ]:
fig = hp.plot_policy_story(runs, ref='Base')
save(fig, 'P_policy_story_2050')

In [ ]:
display(pres.scarcity_numbers(runs))
policy_table(runs, ref='Base')

**The over-dimensioned plant.** The sensitivity represents a plant built for abundant water before a drought: build the electrolysers for abundant water, and then a drought comes.
- In `S2_oversized` the plant is as big as in Base. Water lets it run at no more than 50% of its capacity in any hour, the same water as Medium.
- The difference with Medium is the cost of having built for the optimistic case.

In [ ]:
over = {'Medium (plant sized for the drought)': runs['Medium'], 'Medium, Base plant (oversized)': load('S2_oversized')}
policy_table(over, ref='Medium (plant sized for the drought)')

## 6. Sankey diagrams and the whole energy system

In [ ]:
fig = plot_sankeys_row(runs, period=2050)
save(fig, 'T1_1_sankey_2050')

In [ ]:
save(sp.plot_electricity_supply(runs), 'S_electricity_supply')
save(sp.plot_co2_by_sector(runs), 'S_co2_by_sector')
save(sp.plot_energy_balance(runs['Base'], 'Base: final energy by sector and carrier'), 'S_energy_balance_Base')
save(sp.plot_system_sankey(runs, 2050), 'S_system_sankey_2050')

**CO₂ storage is capped** (10 / 22 / 25 Mt/yr in 2030 / 2040 / 2050). Blue hydrogen and the industrial CCS compete for it, so less electrolysis means more CO₂ from industry.

In [ ]:
pd.concat({k: co2_stored_by_sector(n) for k, n in runs.items()}, axis=1).round(1)

## 7. Variations within the scenarios

**(2a) Cheaper electrolysers:** a subsidy of 20% or 40% of the investment. The limit of Medium stays the same in MW.

In [ ]:
display(pres.subsidy_table())

**(2b) The CO₂ price:** low (170 €/t in 2050), central (250) and high (500), in each scenario.

In [ ]:
co2 = {f'{SHORT[s]}, {p}': (load(f'{s}_co2{p}') if p != 'central' else runs[SHORT[s]])
       for s in ['S1', 'S2', 'S3'] for p in ['low', 'central', 'high']}
compare(co2, period=2050)[['Green (grid electrolysis)', 'Blue (SMR + CCS)', 'Grey (SMR)', 'Electrolysis share (%)',
                           'Electrolyser capacity (GW_el)', 'CO2 total (Mt/yr)', 'System cost (bn €/yr)']]

**(3a–c) Batteries where electrolysis is limited (Medium):** half the cost (3a), and 8 hours instead of 4 (3c). The time series (3b) shows how the electrolysers and batteries run.

In [ ]:
bat = {'Medium': runs['Medium'], 'Medium, half-price batteries': load('S2_battery_cheap'),
       'Medium, 8 h batteries': load('S2_battery_8h')}
display(compare(bat, period=2050)[['Battery (GW)', 'Curtailment (TWh/yr)', 'Electrolyser full-load hours (h/yr)',
                                   'Electrolysis share (%)', 'System cost (bn €/yr)']])
fig = hp.plot_operation_timeseries(runs['Medium'], 2050)
save(fig, 'T1_3b_operation_Medium')
ax = hp.plot_soc_duration({**runs, 'Medium, 8 h': bat['Medium, 8 h batteries']}, 2050)
save(ax, 'T1_3c_soc_duration')

## 7.1. Sensitivity analysis (its own script: `sensitivity.py`)

The sensitivity grids that were in this section now live in `sensitivity.py`, so they can run from the command line (`python sensitivity.py co2_x_subsidy`) or from here. Each grid point is solved once and saved in `results/runs_sens/`; the values go to `results/outputs/sens_<grid>.csv` and the heatmaps to `results/plots/`.

- `co2_x_subsidy`: CO₂ price × electrolyser CAPEX subsidy → electrolysis share (S1, S2, S3)
- `cap_x_co2`: electrolyser cap share (water) × CO₂ price → total CO₂
- `battery_x_co2`: battery cost × CO₂ price → electrolysis share
- freight grids: `bet_vom_x_gasoline_nofcet`, `bet_vom_x_gasoline_h2ice`, `bet_vom_x_gasoline` (the tipping point, section 12.4)


In [ ]:
import sensitivity as sa
RUN_SENS = False            # True = solve the grid points that are not on disk yet
SENS_SOLVER = 'gurobi'
for k, v in sa.GRIDS.items():
    print(f"{k:22s} x={v['x'][0]:24s} y={v['y'][0]:24s} metric={v['metric']}")

In [ ]:
for grid in ['co2_x_subsidy', 'cap_x_co2', 'battery_x_co2']:
    csv = f'results/outputs/sens_{grid}.csv'
    if RUN_SENS:
        df = sa.run_grid(grid, solver=SENS_SOLVER)
    elif os.path.exists(csv):
        df = pd.read_csv(csv)
    else:
        print(f'{grid}: not run yet (RUN_SENS = True, or: python sensitivity.py {grid})')
        continue
    sa.plot_grid(df, grid)
    plt.show()

## 8. Green levers within the water scenarios

How could green policy, technology or subsidies get more green hydrogen when water is scarce? Three levers, each compared with the same scenario without it:
- **Desalinated water** (`desal=True`): an electrolyser on desalinated seawater, **outside** the water limit, at €0.03 per kg extra. `S3_desal_x10` tries €0.30/kg.
- **Production subsidy** (`h2_production_subsidy` = €1 or €2 per kg of electrolytic hydrogen): a negative variable cost of every electrolyser. It is a transfer, so `economic_cost()` adds it back.
- **Mandate** (`green_mandate={2040: 0.6, 2050: 0.6}`): at least 60% of the hydrogen from electrolysis (RED III: 60% renewable hydrogen in industry by 2035). In Medium there is not enough water for it (at most 50% of the demand), so `S2_mandate` has no solution and is not run; in Scarce it is impossible by construction.

In [ ]:
lev = pres.lever_table()          # every lever run against the central run of the same scenario
lev.drop(columns=['lever']).set_index(['scenario', 'label']).round(2)

In [ ]:
for stem in ['P16_green_levers', 'P16b_green_levers_cost_co2']:
    display(Image(f'results/plots/presentation/{stem}.png', width=900))

**Why a subsidy per kg hardly lowers CO₂ (check it here).** The subsidised electrolysers run more hours. The cell below shows where their electricity comes from: the change in what each link takes from its input bus in 2050 (Base with €1/kg minus Base).

In [ ]:
def link_input_2050(n, y=2050):
    w = n.snapshot_weightings.generators.loc[y]
    p0 = n.links_t.p0.loc[y].mul(w, axis=0).sum() / 1e6                    # TWh per year
    return pd.DataFrame({'bus0': n.links.bus0, 'carrier': n.links.carrier, 'TWh': p0}).groupby(['bus0', 'carrier']).TWh.sum()

d = link_input_2050(load('S1_prod1')).sub(link_input_2050(runs['Base']), fill_value=0)
d[d.abs() > 0.3].round(1).to_frame('Base with €1/kg minus Base, 2050 (TWh/yr)')

## 9. Figures for the presentation and the report

`figures/plots/presentation.py` draws every figure from the saved runs, in two styles, both in colour:
- **slides**: sized for the content box of the group's slide template (11.5 × 4.76 inches), in `results/plots/presentation`;
- **APA** (report): 6.5 inches wide, text 8 pt or more, no title or note in the image, in `results/plots/apa`, with the tables `Table_key_numbers` and `Table_green_levers`.

Presentation-ready document diagrams are in `figures_document/`: the model overview, Figures 1–3, and their source scripts. One PNG output is kept for each diagram.

In [ ]:
REDRAW = False                  # True = draw all figures again from the saved runs (both styles, about 20 minutes)
if REDRAW or not os.path.exists('results/plots/presentation/P01_h2_by_route.png'):
    pres.main(styles=('slide', 'apa'))
for stem in ['P01_h2_by_route', 'P02_h2_sankey_2050', 'P03_cost_of_water_scarcity']:
    display(Image(f'results/plots/presentation/{stem}.png', width=900))
display(Image('results/plots/apa/P03_cost_of_water_scarcity.png', width=600))      # the same figure in APA style

## 10. A decision for the group: what is the hydrogen used for?

Francesco set out two cases:
- If the hydrogen is **feedstock** (ammonia, refineries), it is a separate demand on top of the course demands.
- If it is **fuel for high-temperature heat**, the same amount of heat must come off the industrial HTP demand.

Our 1.3 Mt is today's feedstock demand, so the default is `'feedstock'`. `S1_htp_fuel` shows the other choice: the industrial HTP demand is reduced by the heat that this hydrogen would deliver in a furnace.

In [ ]:
use = {'Feedstock (on top)': runs['Base'], 'Fuel for heat (HTP reduced)': load('S1_htp_fuel')}
print('HTP reduction (TWh/yr):', use['Fuel for heat (HTP reduced)'].meta.get('htp_reduction_twh'))
pd.concat({p: compare(use, period=p)[['Electrolysis share (%)', 'Gas use, total (TWh/yr)', 'CO2 total (Mt/yr)',
                                      'System cost (bn €/yr)']] for p in [2020, 2050]})

## 11. What is not in the model

- **Water.** There is no water resource; the scenarios limit electrolyser capacity instead. Water use is reported afterwards, in litres per kg: electrolysis 22, SMR 10, SMR + CCS 28.
- **The hydrogen network.** Left out to keep the model simple. Hydrogen storage is in the package with additions.
- **Hydrogen imports and an endogenous hydrogen demand** (hydrogen chosen by the model, e.g. for heat). These are not our question: the demand is given, and the scenarios differ only in water.

## 5. Reproducible batch run from the interface

The next cell uses the exact same `RUNS` registry and `run()` function as `run_batch.py`.


In [ ]:
# Run exactly the same scenario registry as run_batch.py.
# Set RUN_ALL_SCENARIOS=True and execute this cell to solve every configured run.
RUN_ALL_SCENARIOS = False
RERUN = False  # True forces a fresh solve; False reuses results/runs/*.nc


def run_all_scenarios(rerun=RERUN):
    results = {}
    for name in run_batch.RUNS:
        path = run_batch.run(name, solver=SOLVER, rerun=rerun)
        results[name] = path
    solved = [name for name, path in results.items() if path]
    missing = [name for name, path in results.items() if not path]
    print(f'Solved or loaded {len(solved)}/{len(results)} configured scenarios')
    if missing:
        print('Not saved (normally infeasible):', ', '.join(missing))
    return results


if RUN_ALL_SCENARIOS:
    RUN_RESULTS = run_all_scenarios()


In [ ]:
# Optional: regenerate figures from the saved runs in this package.
# This is intentionally separate from solving so the notebook can inspect or plot existing results.
RUN_ALL_FIGURES = False
if RUN_ALL_FIGURES:
    if 'pres' in globals():
        pres.main(styles=('slide', 'apa'))
    if 'sp' in globals():
        sp.main()


# 12. Freight study: trucks in transition (individual assignment)

Heavy-duty road freight is a new demand of 90 PJ in 2020 (energy terms, GDP driver), served by ICE trucks (gasoline) and battery-electric trucks (BET): the assignment's core (`_nofcet` runs). The hydrogen extension adds two hydrogen trucks fed by the group's hydrogen module: fuel-cell trucks (FCET) and hydrogen combustion trucks (H2-ICE truck, `h2ice`). `build_freight_database.py` writes `input_databases/freight_nl`; `run_freight.py` holds the runs.

| Run | BET costs (X) | Gasoline price (Y) |
|---|---|---|
| F1 | constant | moderate |
| F2 | decrease | moderate |
| F3 | constant | sharp |
| F4 | decrease | sharp |

All runs: moderately increasing CO₂ price (`co2_price='low'`). Freight demand follows GDP (elasticity 0.6, the macroeconomic driver); water scarcity is an outside driver in the `_real_water` runs only (section 12.2). Variants: `_ice` (no zero-emission trucks: the cost of switching), `_nofcet`, `_co2central`, `_co2high`, `_sunday_ban`, `_night_ban`, the calibration tests (group `tests`, below) and the realistic layer (`_real`, section 12.5). Run groups: `python run_freight.py --list`; figures of the paper: `python analysis_freight.py` and `python figure_set.py --redraw`.

In [ ]:
import run_freight, build_freight_database
from figures.plots import freight_results as fr
FREIGHT_SOLVER = 'gurobi'
RUN_FREIGHT = False          # True = solve runs that are not on disk yet
if not os.path.exists('input_databases/freight_nl/technologies.xlsx'):
    build_freight_database.main()

def load_f(name):
    path = os.path.join('results', 'runs', f'{name}.nc')
    if RUN_FREIGHT or not os.path.exists(path):
        path = run_freight.run(name, solver=FREIGHT_SOLVER)
    n = pypsa.Network(path); n.name = name
    return n

F = {f: load_f(f) for f in ['F1', 'F2', 'F3', 'F4']}
fr.compare(F).round(2)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 6))
fr.plot_freight_mix(F, ax=axes[0]); fr.plot_co2(F, ax=axes[1])
plt.tight_layout(); save(fig, 'F_freight_mix_co2')

### Calibration and tests (assignment step 5)

The base year (2020) is the existing stock only (`ICE truck - existing`, `BET - existing` 0.1 GW), the same in every scenario. The tests (`python run_freight.py --group tests`) check this stock and the inputs behind it: the base year without the 'existing' trucks (`_nostock`: not calibrated, 99.95% BETs in 2020), the 90 PJ read as fuel input (`_fuel90`), real diesel prices (`_diesel`), fuel without excise (`_untaxed`), the course coordinator's high charging penalty (`_vom3`) and 6-hour time steps (`F1_tres6`). Table: `results/outputs/freight_calibration_tests.csv`.

In [ ]:
cal = {k: load_f(k) for k in run_freight.GROUPS['core'][:4] + run_freight.GROUPS['tests']}
cols = ['Freight demand (TWh/yr)', 'BET share of freight (%)', 'ZE share of freight (%)', 'CO2 freight trucks (Mt/yr)']
display(fr.compare(cal)[cols].unstack('period').round(1))

## 12.1 The cost of switching to zero-emission freight

Each scenario against its counterfactual without new zero-emission trucks (`_ice`: no new BETs, FCETs or H2-ICE trucks).

In [ ]:
pd.concat({f: fr.switching_cost(F[f], load_f(f'{f}_ice')) for f in F}).round(2)

## 12.2 Water use and the water-scarcity driver

`fr.water_account(n)` gives the water and freshwater consumption of **every technology** in the system (output × intensity in `energy_model/freight_presets.py: WATER_INTENSITY`; industrial CCS per t CO₂ captured). Water scarcity is not a scenario axis but an **outside driver**, like GDP for the demands (`water_scarcity='driver'`): in the dry season (June–September) each **verdringingsreeks** category may use a share of the water it used in the reference run F1 (power plants category 2, process water 3, electrolysis 3 or 4 via `electrolysis_category`, PV 4). The path tightens over time: the Deltaprogramma's 1-in-20 dry year in 2030 and 2040 (category 4 at 50%) and a 2018-type drought in 2050 (category 3 at 75%, category 4 at 0%), with summer derating of river-cooled plants (10 / 15 / 25%). Water above a quota is desalinated or reused water at `water_alt_cost` (€1.4/m³). `fr.drought_table(n)` shows use, quota and alternative water per category. Runs: `F1_real_water`, `F4_real_water`, `F1_real_water_ely4` (electrolysis in category 4).

In [ ]:
acc = fr.water_account(F['F1'])
display(acc.pivot_table(index='technology', columns='period', values='water_Mm3', aggfunc='sum').round(2))
display(fr.water_summary(F['F1']).round(2))
display(pd.concat({f: fr.freight_water(n) for f, n in F.items()}).round(2))

In [ ]:
W = {'F1 realistic': load_f('F1_real'), 'F1 realistic, water driver': load_f('F1_real_water'),
     'F1 realistic, water driver, electrolysis cat. 4': load_f('F1_real_water_ely4')}
display(fr.compare(W, period=2050).T.round(2))
for k in list(W)[1:]:
    display(fr.drought_table(W[k]).round(2).rename_axis(k))
ax = fr.plot_water(W, period=2050); save(ax, 'F_water_2050')

## 12.3 Shadow prices and storage

`store_shadow_prices` (restored) saves the duals of the extra constraints on `n.meta` after every solve; values are per year of the period.
- **Water:** the drought-quota dual (per verdringingsreeks category) is the value of one more m³ of freshwater in the drought season (at most the alternative-water price, 1.4 €/m³).
- **Other limits:** net-zero CO₂ cap and CO₂-storage cap (€/t), truck and RFNBO mandates, build limits (offshore wind, nuclear; €/kW/yr).
- **Storage:** grid batteries (GW, GWh, hours, cycles) and salt-cavern hydrogen storage (TWh). `F1_tres6` (a test) repeats F1 at 6-hour steps.


In [ ]:
on_disk = lambda k: os.path.exists(os.path.join('results', 'runs', f'{k}.nc'))
for k in ['F1_real', 'F1_real_water', 'F1_real_water_ely4', 'F1_real_netzero', 'F4_real_netzero']:
    if on_disk(k):
        sp = fr.shadow_prices(load_f(k)).drop(index=2020, errors='ignore')
        display(sp.loc[:, sp.abs().max() > 1e-3].round(2).T.rename_axis(k))
display(pd.concat({k: fr.storage(load_f(k)) for k in ['F1_nofcet', 'F1', 'F1_tres6']
                   if on_disk(k)}).round(2))

## 12.4 Freight sensitivities

The assignment's sensitivities (carbon price, time restrictions) are single runs in `run_freight.py` (groups `core_sensitivity` and `real_sensitivity`; heatmap `R5_sensitivities`). The grids in `sensitivity.py` map the tipping point behind them:
- `bet_vom_x_gasoline_nofcet`, `bet_vom_x_gasoline_h2ice`, `bet_vom_x_gasoline`: where ICE trucks stay competitive (charging penalty × fuel price), with ICE + BET, + H2-ICE truck, and + H2-ICE truck + FCET

In [ ]:
for grid in ['bet_vom_x_gasoline_nofcet', 'bet_vom_x_gasoline_h2ice', 'bet_vom_x_gasoline']:
    csv = f'results/outputs/sens_{grid}.csv'
    if RUN_SENS:
        df = sa.run_grid(grid, solver=FREIGHT_SOLVER)
    elif os.path.exists(csv):
        df = pd.read_csv(csv)
    else:
        print(f'{grid}: not run yet (RUN_SENS = True, or: python sensitivity.py {grid})')
        continue
    sa.plot_grid(df, grid)
    plt.show()

## 12.5 Realistic layer: current policies, real prices and all four truck types

The same four scenarios (`F1_real` … `F4_real`) under the truck and hydrogen policies in force or planned in October 2026 (EU CO2 standards as a floor on the zero-emission share, the truck toll with AanZET, RED III transport, the Dutch industry obligation for renewable hydrogen, OWE and SWiM subsidies, AFIR roll-out), with real prices (diesel excluding VAT on both Y paths) and market frictions (BETs by duty cycle, a heavy special segment, the market ramp-up, hydrogen-truck availability). Twins: `_real_noh2` (ICE + BET), `_real_ice` (ICE only, for the cost of switching), `_real_nopolicy` (no truck or hydrogen policy). Settings: `REAL_FRICTIONS`, `TRUCK_POLICIES`, `H2_POLICIES` in `run_freight.py`; values and sources in `energy_model/freight_presets.py` ('Realistic layer'). All figures of the paper: `python analysis_freight.py`.

In [ ]:
from figures.plots import freight_system as fsys
R = {f: load_f(f'{f}_real') for f in ['F1', 'F2', 'F3', 'F4']}
cols = [c for c in fr.compare(R).columns if 'share of freight' in c or 'CO2' in c]
display(fr.compare(R)[cols].round(1))
display(fr.policy_transfers(R['F1']).round(2))       # toll paid and subsidies: transfers next to the system cost
fig = fsys.plot_sankey({'F1 core': load_f('F1_nofcet'), 'F1 realistic, current policies': R['F1']}, period=2050)
save(fig, 'F_sankey_2050')